In [7]:
import pathlib
import re

from train import get_trained_classifier, process_file_data
from sklearn.metrics import accuracy_score
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

In [2]:
participants_dict = {
    'KrpKfx': [4,2,6],
    'uVoAnv': [2,4,6],
    '2iTyoP': [6,2,4],
    '9Z0czI': [2,6,4],
    'on7sut': [4,6,2],
    'yX98eE': [6,4,2],
    'DO3yNb': [2,4,6],
    'EOx0vb': [4,2,6],
    'zUmr0W': [6,2,4],
    '3H1N53': [2,6,4],
    '3SePwH': [4,6,2],
    'O8gX1A': [6,4,2]
}

# Average Response Time

In [ ]:
def add_to_dataframe(total_df, folder):
    print(folder_key)
    for part in participants_dict.keys():
        file_dict = dict()
        for file in folder.glob(f'{part}*.txt'):
            s = open(file, 'r').read()
            end = re.search('Trial End', s)
            if end is not  None:
                match = re.search('\n POSES:.*\n', s)
                if match is None:
                    raise Exception('file doesnt contain poses')
                pose_list = [int(x) for x in
                            match.group().split(':')[-1].strip()[1:-1].split(', ')]
                num_pose = len(pose_list)
                file_dict[num_pose] = file

        df_list = list()
        for num_pose, file in file_dict.items():
            df = pd.DataFrame()
            x, y, trial_length = process_file_data(file)
            df['trial_length'] = trial_length
            df['num_pose'] = num_pose
            df['target_pose'] = y
            df['participant_id'] = part
            df['position'] = participants_dict[part].index(num_pose) + 1
            df['type'] = folder_key
            df_list.append(df)

        df = pd.concat(df_list)
        total_df = pd.concat([total_df, df])
        print(part)
    return total_df

In [ ]:
folder_key = pathlib.Path('.').parent / 'keyboard-results'
total_list = add_to_dataframe(pd.DataFrame(), folder_key)

folder_key = pathlib.Path('.').parent / 'prompt-results'
total_list = add_to_dataframe(total_list, folder_key)

In [ ]:
total_list['time (sec)'] = total_list['trial_length'] / 30

In [ ]:
df_mean_per_num_pose = total_list.groupby(['type','participant_id', 'num_pose']).mean()

In [ ]:
sns.scatterplot(data=df_mean_per_num_pose, y='time (sec)', x='num_pose', hue='participant_id', style='type', s = 100, palette='Paired')

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xticks([2, 4, 6])
plt.title("Average Response Time Per # Poses")

plt.show()

In [ ]:
def plt_one_participant(p_id):
    data = total_list[total_list["participant_id"]==p_id].groupby(['type','participant_id', 'num_pose']).mean()
    sns.scatterplot(data=data, y='time (sec)', x='num_pose', hue='participant_id', style='type', s = 100, palette="Paired")

    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xticks([2, 4, 6])
    plt.title('Average Response Time For {}, Order: {}'.format(p_id, participants_dict[p_id]))

    plt.show()

In [ ]:
# plt_one_participant("DO3yNb")
# plt_one_participant("uVoAnv")

In [ ]:
twos = total_list[total_list["num_pose"]==2].groupby(['type','participant_id', 'num_pose']).mean()

sns.scatterplot(data=twos, y='time (sec)', x='position', hue='participant_id', style='type', s = 100, palette="Paired")

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xticks([1, 2, 3])
plt.title("Average Response Time For 2 Poses")

plt.show()

fours = total_list[total_list["num_pose"]==4].groupby(['type','participant_id', 'num_pose']).mean()

sns.scatterplot(data=fours, y='time (sec)', x='position', hue='participant_id', style='type', s = 100, palette="Paired")

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xticks([1, 2, 3])
plt.title("Average Response Time For 4 Poses")


plt.show()

sixes = total_list[total_list["num_pose"]==6].groupby(['type','participant_id', 'num_pose']).mean()

sns.scatterplot(data=sixes, y='time (sec)', x='position', hue='participant_id', style='type', s = 100, palette="Paired")

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xticks([1, 2, 3])
plt.title("Average Response Time For 6 Poses")

plt.show()

In [ ]:
sns.scatterplot(data=df_mean_per_num_pose, y='time (sec)', x='position', hue='participant_id', style='type', s = 100, palette="Paired")

plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.xticks([1, 2, 3])
plt.title("Average Response Time For Each Round")

plt.show()

# Accuracy - WIP

In [4]:
def create_classifier_dict():
    class_dict = {}
    for part in participants_dict.keys():
        class_dict[part] = get_trained_classifier(part)
    return class_dict


classifier_dict = create_classifier_dict()
classifier_dict

{'KrpKfx': <norm_class.NormClass at 0x161018b50>,
 'uVoAnv': <norm_class.NormClass at 0x14414c430>,
 '2iTyoP': <norm_class.NormClass at 0x160e09e80>,
 '9Z0czI': <norm_class.NormClass at 0x160e099a0>,
 'on7sut': <norm_class.NormClass at 0x1611580d0>,
 'yX98eE': <norm_class.NormClass at 0x16109b0d0>,
 'DO3yNb': <norm_class.NormClass at 0x160df66a0>,
 'EOx0vb': <norm_class.NormClass at 0x160df6520>,
 'zUmr0W': <norm_class.NormClass at 0x160c2f790>,
 '3H1N53': <norm_class.NormClass at 0x16134df70>,
 '3SePwH': <norm_class.NormClass at 0x146657a90>,
 'O8gX1A': <norm_class.NormClass at 0x160b84520>}

In [23]:
def accuracy_dataframe(total_df, folder):
    print(folder_key)
    for part in list(participants_dict.keys())[0:2]:
        file_dict = dict()
        for file in folder.glob(f'{part}*.txt'):
            s = open(file, 'r').read()
            end = re.search('Trial End', s)
            if end is not  None:
                match = re.search('\n POSES:.*\n', s)
                if match is None:
                    raise Exception('file doesnt contain poses')
                pose_list = [int(x) for x in
                            match.group().split(':')[-1].strip()[1:-1].split(', ')]
                num_pose = len(pose_list)
                file_dict[num_pose] = file

        poses = list()
        accs = list()
        ids = list()
        pos = list()
        types = list()
        
        for num_pose, file in file_dict.items():
            x, y, trial_length = process_file_data(file)
            print(y)
            
            y_pred = classifier_dict[part].predict(x) 
            print(y_pred)
            
            poses.append(num_pose)
            accs.append(accuracy_score(y_pred=y_pred, y_true =y))
            ids.append(part)
            pos.append(participants_dict[part].index(num_pose) + 1)
            types.append(folder_key)
        
        df = pd.DataFrame()
        df['num_pose'] = poses
        df['accuracy'] = accs
        df['participant_id'] = ids
        df['position'] = pos
        df['type'] = types

        total_df = pd.concat([total_df, df])
        print(part)
    return total_df

In [24]:
folder_key = pathlib.Path('.').parent / 'keyboard-results'
accuracy_dataframe(pd.DataFrame(), folder_key)

keyboard-results
[3 3 2 3 5 3 0 0 4 1 2 5 3 1 3 4 1 2 2 4 4 0 4 0 2 0 2 0 0 1 2 0 5 4 1 0 5
 3 4 5 4 3 1 0 0 2 4]
[5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 4 5 5 5]
[5 5 1 5 0 5 2 2 5 1 1 0 5 1 5 5 1 1 1 0 0 2 5 2 1 2 2 2 2 1 2 2 0 5 1 2 0
 5 5 0 5 5 1 2 2 2 5]
[5 5 2 5 2 5 5 5 5 5 5 5 3 5 5 5 5 2 2 5 5 5 5 3 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5]
[2 2 1 2 2 2 1 1 2 1 1 2 2 1 2 2 1 1 1 2 2 1 2 1 1 1 1 1 1 1 1 1 2 2 1 1 2
 2 2 2 2 2 1 1 1 1 2]
[5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 1 1 5 5 5 5 5 5 5 1 5 5 5 5 1 5 5 5 5 5
 5 5 5 5 1 1 5 5 5 1]
KrpKfx
[5 5 2 5 5 5 2 2 5 2 2 5 5 2 5 5 2 2 2 5 5 2 5 2 2 2 2 2 2 2 2 2 5 5 2 2 5
 5 5 5 5 5 2 2 2 2 5]
[5 5 5 2 5 5 5 2 2 2 4 2 5 5 2 5 5 2 2 2 2 5 1 2 2 2 2 2 2 2 2 2 2 5 2 2 2
 2 5 5 2 2 2 2 2 2 2]
[0 0 2 0 1 0 5 5 0 2 2 1 0 2 0 0 2 2 2 1 1 5 0 5 2 5 5 5 5 2 5 5 1 0 2 5 1
 0 0 1 0 0 2 5 5 5 0]
[2 1 1 2 1 2 2 5 2 1 2 2 2 1 2 1 1 2 2 2 2 1 1 1 2 2 2 2 2 2 2 2 2 1 1 2 5
 2 1 1 2 1 1 2 2 2 2]
[3 3 5

,num_pose,accuracy,participant_id,position,type
0,6,0.106383,KrpKfx,3,keyboard-results
1,4,0.297872,KrpKfx,1,keyboard-results
2,2,0.106383,KrpKfx,2,keyboard-results
0,2,0.595745,uVoAnv,1,keyboard-results
1,4,0.170213,uVoAnv,2,keyboard-results
2,6,0.148936,uVoAnv,3,keyboard-results
